# <center> **Playground: Intents model**

This notebook is intended to test an `intents` text-classification model based on `distilbert base uncased`.

---

In [1]:
!pip -q install requests tokenizers pydantic requests


[notice] A new release of pip available: 22.2.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


## Set Up

In [2]:
# Notebook arguments
api_url = 'http://modelmesh-serving.ap1001-analytics:8008/v2/models/banorte-intents/infer'
token = ''
label_map_file = 'id2label.json'

In [3]:
# Classes
import torch
from tokenizers import Tokenizer
from pydantic import BaseModel
import requests

class Label(BaseModel):
    label: str
    score: float

    @classmethod
    def make_from_tensor(cls, tensor: torch.tensor, label_map):
        tensor_2d = tensor.unsqueeze(0)
        probabilities = torch.nn.functional.softmax(tensor_2d, dim=1)
        predicted_class_index = probabilities.argmax(dim=1).tolist()
        predicted_class_score = probabilities[0, predicted_class_index].item()
        return cls(label=label_map[str(predicted_class_index[0])], score=predicted_class_score)


class OpenVINOTextClassificationModel():
    token_model = "distilbert-base-uncased"
    model_cls = Label

    def __init__(self, url, token, label_map={}):
        self.url = url
        self.label_map = label_map
        self.tokenizer = Tokenizer.from_pretrained(self.token_model)

    def payload(self, tquery):
        ids = tquery.ids
        am = tquery.attention_mask
        body = {
            "inputs": [
                {
                    "name": "input_ids",
                    "shape": [1, len(ids)],
                    "datatype": "INT64",
                    "data": [ids],
                },
                {
                    "name": "attention_mask",
                    "shape": [1, len(am)],
                    "datatype": "INT64",
                    "data": [am],
                },
            ]
        }
        return body

    def transform(self, tensor):
        return self.model_cls.make_from_tensor(tensor, self.label_map)

    def infer(self, query, **kwargs):
        encoded = self.tokenizer.encode(query)
        body = self.payload(encoded)
        response = requests.post(self.url, json=body)
        response = response.json()
        logits = response.get("outputs")[0]["data"]
        tensor = torch.tensor(logits)
        return self.transform(tensor)

In [4]:
# Instantiation
import json

label_map = {}
with open(label_map_file, 'r') as _f:
    label_map = json.load(_f)

model = OpenVINOTextClassificationModel(api_url, token, label_map)

print(label_map)

{'0': 'spam', '1': 'conversational', '2': 'irrelevant'}


## Testing
Update the next inputs to call the model and show its results! intented to serve as an exam to see how it goes from a functional standpoint

In [5]:
test_inputs = [
  ('Cuando podria ver la proxima luna llena desde CDMX?', 'irrelevant'),
  ('askfjhnwekljfh', 'spam'),
  ('Puedes explicar resumidamente los estandares de desarrollo de software en Banorte?', 'conversational'),
  ('Para que se utilizan los logical requirements en Banorte?', 'conversational'),
  ('Quien es el presidente de los estados unidos?', 'irrelevant'),
  ('¿Cuál es la importancia de diversificar tu cartera de inversiones para reducir el riesgo en el mercado financiero?', 'irrelevant')
]

for user_input, expected_label in test_inputs:
    response = model.infer(user_input)
    print(response)
    assert(response.label == expected_label)

label='irrelevant' score=0.9874052405357361
label='spam' score=0.9978184700012207
label='conversational' score=0.9978165626525879
label='conversational' score=0.996887743473053
label='irrelevant' score=0.9970963001251221
label='irrelevant' score=0.997424840927124


## Playground
Write your input and see the response in real time

In [6]:
import json
import os

# Function to load existing data from the file
def load_data(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as file:
            return json.load(file)
    return []

# Function to save new data to the file
def save_data(data, file_path):
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

In [8]:
# Load existing data
file_path = 'user-inputs/data.json'
existing_data = load_data(file_path)

# Start a loop to collect multiple user inputs
while True:
    # Collect user input
    user_data = {}
    user_data['text'] = input("Enter \033[1mtext\033[0m to be evaluated:\n")
    user_data['user-input'] = input("Enter \033[1mexpected label\033[0m:\n")

    # Collect model output
    response = model.infer(user_data['text'])
    user_data['model-pred'] = response.label
    user_data['model-score'] = response.score
    print(f"Intents \033[1mmodel output\033[0m: {response}")
    
    # Append the new data to the existing data
    existing_data.append(user_data)

    # Ask if the user wants to continue entering data
    continue_input = input("Do you want to \033[1mcontinue?\033[0m (yes/no): ").strip().lower()
    print("-"*25)
    if continue_input != 'yes':
        break
        
# Save all the collected data to the JSON file
save_data(existing_data, file_path)
print(f"All data saved successfully in {file_path} :)")

Enter text to be evaluated:
 is banorte a member of sgt pepper lonely hearts club band?
Enter expected label:
 irrelevant


Intents model output: label='spam' score=0.6115590929985046


Do you want to continue? (yes/no):  yes


-------------------------


Enter text to be evaluated:
 saj u23o 8127
Enter expected label:
 spam


Intents model output: label='spam' score=0.997215747833252


Do you want to continue? (yes/no):  can i have some tea?


-------------------------
All data saved successfully in user-inputs/data.json :)


---